In [1]:
# Ensure project root on sys.path for `src` imports
import sys
from pathlib import Path

cwd = Path.cwd()
for base in [cwd, cwd.parent, cwd.parent.parent]:
    if (base / "src").exists():
        sys.path.insert(0, str(base))
        break


# Student Guide: Retrieval

Goal: find the most relevant chunks for a user query.

What you’ll learn:
- Cosine similarity for ranking documents.
- Hybrid retrieval: combining sparse TF–IDF with dense LSA.
- How `top‑k` changes recall vs. latency.

Algorithm overview:
1) Vectorize the query with the fitted TF–IDF (and SVD for dense).
2) Compute cosine scores vs. the corpus (sparse and/or dense).
3) Combine scores with weights `w_sparse` and `w_dense`.
4) Return the top‑k with texts and component scores.

References:
- Cosine similarity: https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity
- Hybrid search overview: https://www.pinecone.io/learn/hybrid-search/
- Query embedding with TF–IDF: https://scikit-learn.org/stable/modules/feature_extraction.html#tfidf-term-weighting

Try this:
- Tune `retrieval.weights` in `config/settings.yaml`.
- Increase `k` and observe how results change.


# 03 - Retrieval

Query the index and inspect the top-K retrieved snippets.



In [2]:
from src.retrieve import retrieve

QUERY = "Where can I see Macbeth in Colorado this summer?"
MAX_ITEMS = 5
SNIPPET = 200

results = retrieve(QUERY, k=max(20, MAX_ITEMS))

def show(results, max_items=5, max_chars=200):
    for i, r in enumerate(results[:max_items], 1):
        text = (r.get("text", "") or "").replace("\n", " ")
        if len(text) > max_chars:
            text = text[:max_chars] + "..."
        score = r.get("score")
        origin = r.get("origin")
        idx = r.get("index")
        head = f"{i}. {round(score, 3) if isinstance(score, (int, float)) else ''} {text}"
        tail = f"\n   -> {origin} [#{idx}]" if origin else ""
        print(head + tail)

show(results, MAX_ITEMS, SNIPPET)


1. 0.008 *** START OF THE PROJECT GUTENBERG EBOOK 100 *** The Complete Works of William Shakespeare by William Shakespeare Contents THE SONNETS ALL’S WELL THAT ENDS WELL THE TRAGEDY OF ANTONY AND CLEOPATRA AS ...
2. 0.008 *** START OF THE PROJECT GUTENBERG EBOOK 100 *** The Complete Works of William Shakespeare by William Shakespeare Contents THE SONNETS ALL’S WELL THAT ENDS WELL THE TRAGEDY OF ANTONY AND CLEOPATRA AS ...


# 03 – Retrieval

Retrieve top‑K relevant chunks for a user query using cosine similarity.


In [3]:
from pathlib import Path

from src.ingest import load_raw_documents
from src.embed import train_tfidf, embed_query
from src.retrieve import retrieve_top_k

# Load documents
processed_dir = "data/processed"
raw_dir = "data/raw"
if not any(Path(processed_dir).glob("*.txt")):
    documents = load_raw_documents(raw_dir)
else:
    documents = [p.read_text(encoding="utf-8") for p in Path(processed_dir).glob("*.txt")]

# Train and query
vectorizer, matrix = train_tfidf(documents)
q = "Where can I see Macbeth in Colorado this summer?"
qv = embed_query(q, vectorizer)
results = retrieve_top_k(qv, matrix, documents, top_k=5)
for i, (idx, score, text) in enumerate(results, start=1):
    print(i, round(score, 3), text[:200].replace("\n", " ") + ("..." if len(text) > 200 else ""))


1 0.402 Hear it not, Duncan, for it is a knell That summons thee to heaven or to hell. [_Exit._] SCENE II. The same. Enter Lady Macbeth. LADY MACBETH. That which hath made them drunk hath made me bold: What h...
2 0.402 Hear it not, Duncan, for it is a knell That summons thee to heaven or to hell. [_Exit._] SCENE II. The same. Enter Lady Macbeth. LADY MACBETH. That which hath made them drunk hath made me bold: What h...
3 0.337 If there come truth from them (As upon thee, Macbeth, their speeches shine) Why, by the verities on thee made good, May they not be my oracles as well, And set me up in hope? But hush; no more. Sennet...
4 0.337 If there come truth from them (As upon thee, Macbeth, their speeches shine) Why, by the verities on thee made good, May they not be my oracles as well, And set me up in hope? But hush; no more. Sennet...
5 0.296 A foolish thought, to say a sorry sight. MACBETH. There’s one did laugh in’s sleep, and one cried, “Murder!” That they did wake each other: I st